In [ ]:
from pathlib import Path
# ---- x8-panel-fps: kit fingerprints (features.structure, Morgan r2 2048) of every candidate in Saner's labelled panel ----
import subprocess, sys, numpy as np, pandas as pd
try:
    from rdkit import rdBase
except ImportError:
    wheel = sorted(Path('/kaggle/input').rglob('rdkit-2026.3.3-*.whl'))[0]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', str(wheel.parent), 'rdkit==2026.03.3'], check=True)
    from rdkit import rdBase
print('RDKIT', rdBase.rdkitVersion)
INP = Path('/kaggle/input')
KIT = next(p.parent for p in INP.rglob('features.py') if (p.parent / 'final_model.pt').is_file())
sys.path.insert(0, str(KIT)); import features
panel = pd.read_parquet(next(INP.rglob('labelled_candidate_scores.parquet')))
smiles = panel.smiles.astype(str).unique()
keys, fps, bad = [], [], []
for s in smiles:
    try:
        k, _, _, _, fp = features.structure(s, 2048); keys.append(k); fps.append(np.asarray(fp, dtype=np.uint8))
    except Exception as e:
        bad.append((s, repr(e)[:80])); keys.append(''); fps.append(np.zeros(2048, np.uint8))
np.savez_compressed('/kaggle/working/panel_fps.npz', smiles=np.array(smiles), keys=np.array(keys), fp=np.stack(fps))
print('FPS', len(smiles), 'bad', len(bad), bad[:5])
